In [1]:
import os
import polars as pl
import gc
import shutil
from tqdm.auto import tqdm

SASREC_CAND_PATH = '/kaggle/input/datasets/b22dckh072/file04/sasrec_candidates.parquet'
LIGHTGCN_CAND_PATH = '/kaggle/input/datasets/b22dckh072/file04/lightgcn_candidates.parquet'
TEMP_DIR = '/kaggle/working/candidates_chunks_temp'
FINAL_CAND_PATH = '/kaggle/working/final_combined_candidates.parquet'

In [2]:
# 1. Tính toán Top 50 sản phẩm phổ biến nhất từ tập Train
print("Đang tính toán danh sách sản phẩm phổ biến (Popularity)...")
TRAIN_PATH = '/kaggle/input/datasets/b22dckh072/file04/train_interactions.parquet'
top_popular_items = (
    pl.scan_parquet(TRAIN_PATH)
    .group_by('mapped_item_id')
    .len()
    .sort('len', descending=True)
    .head(50) # Lấy top 50 sản phẩm
    .select('mapped_item_id')
    .collect()
)

# Chuyển thành list để sử dụng trong join
popular_items_list = top_popular_items['mapped_item_id'].to_list()
print(f"-> Đã chọn {len(popular_items_list)} sản phẩm phổ biến làm dự phòng.")

Đang tính toán danh sách sản phẩm phổ biến (Popularity)...
-> Đã chọn 50 sản phẩm phổ biến làm dự phòng.


In [3]:
os.makedirs(TEMP_DIR, exist_ok=True)

if os.path.exists(SASREC_CAND_PATH) and os.path.exists(LIGHTGCN_CAND_PATH):
    print("Đang quét cấu trúc file (Lazy Scan)...")
    
    lazy_sasrec = pl.scan_parquet(SASREC_CAND_PATH)
    lazy_lightgcn = pl.scan_parquet(LIGHTGCN_CAND_PATH)
    
    # NẠP TẬP TRAIN ĐỂ LÀM MÀNG LỌC (BẠN ĐÃ QUÊN DÒNG NÀY)
    lazy_train = pl.scan_parquet('/kaggle/input/datasets/b22dckh072/file04/train_interactions.parquet')

    print("Đang tính toán số lượng người dùng...")
    max_u_sasrec = lazy_sasrec.select(pl.col('mapped_user_id').max()).collect().item()
    max_u_lightgcn = lazy_lightgcn.select(pl.col('mapped_user_id').max()).collect().item()
    
    max_u_sasrec = 0 if max_u_sasrec is None else max_u_sasrec
    max_u_lightgcn = 0 if max_u_lightgcn is None else max_u_lightgcn
    max_user = max(max_u_sasrec, max_u_lightgcn)

    chunk_size = 50000 
    
    print(f"Tổng số User ID: {max_user}. Bắt đầu gộp và chia nhỏ file...")

    for start_u in tqdm(range(0, max_user + 1, chunk_size), desc="Đang xử lý từng phần"):
        end_u = start_u + chunk_size

        chunk_sasrec = lazy_sasrec.filter(
            (pl.col('mapped_user_id') >= start_u) & (pl.col('mapped_user_id') < end_u)
        ).collect()

        chunk_lightgcn = lazy_lightgcn.filter(
            (pl.col('mapped_user_id') >= start_u) & (pl.col('mapped_user_id') < end_u)
        ).collect()

        if chunk_sasrec.height == 0 and chunk_lightgcn.height == 0:
            continue

        chunk_union = chunk_sasrec.join(
            chunk_lightgcn, 
            on=['mapped_user_id', 'mapped_item_id'], 
            how='full', 
            coalesce=True
        )
        
        current_users = chunk_union['mapped_user_id'].unique()
        df_pop_all_users = pl.DataFrame({
            'mapped_user_id': current_users.to_list()
        }).join(top_popular_items, how='cross') 

        chunk_union = chunk_union.join(
            df_pop_all_users, 
            on=['mapped_user_id', 'mapped_item_id'], 
            how='full', 
            coalesce=True
        )
        
        chunk_train = lazy_train.filter((pl.col('mapped_user_id') >= start_u) & (pl.col('mapped_user_id') < end_u)).collect()
        chunk_union = chunk_union.join(
            chunk_train.select(['mapped_user_id', 'mapped_item_id']), 
            on=['mapped_user_id', 'mapped_item_id'], 
            how='anti'
        )
        chunk_union = (
            chunk_union
            .with_columns([
                (pl.col('sasrec_rank').fill_null(250) + pl.col('lightgcn_rank').fill_null(250)).alias('borda_rank')
            ])
            .sort(['mapped_user_id', 'borda_rank', 'lightgcn_rank'])
            .group_by('mapped_user_id', maintain_order=True)
            .head(100)
            .drop('borda_rank') 
        )

        chunk_file_path = f"{TEMP_DIR}/cand_{start_u}_to_{end_u}.parquet"
        chunk_union.write_parquet(chunk_file_path)

        del chunk_sasrec, chunk_lightgcn, chunk_union, chunk_train
        gc.collect()

    print("Đang hợp nhất các khối thành 1 file duy nhất ")
    pl.scan_parquet(f"{TEMP_DIR}/*.parquet").sink_parquet(FINAL_CAND_PATH)
    shutil.rmtree(TEMP_DIR)
    
    print(f"File tại: {FINAL_CAND_PATH}")

else:
    print("Lỗi: Không tìm thấy file đầu vào. Hãy kiểm tra lại đường dẫn.")

Đang quét cấu trúc file (Lazy Scan)...
Đang tính toán số lượng người dùng...
Tổng số User ID: 2257153. Bắt đầu gộp và chia nhỏ file...


Đang xử lý từng phần:   0%|          | 0/46 [00:00<?, ?it/s]

Đang hợp nhất các khối thành 1 file duy nhất 
File tại: /kaggle/working/final_combined_candidates.parquet


In [4]:
import os
from IPython.display import FileLink

# Thay 'your_output_file.csv' bằng tên file bạn muốn tải
filename ='/kaggle/working/final_combined_candidates.parquet'

if os.path.exists(filename):
    display(FileLink(filename))
else:
    print("Không tìm thấy file. Hãy kiểm tra lại tên file trong thư mục /kaggle/working/")

/kaggle/working/final_combined_candidates.parquet